# Lab 3 — The desk in ADK, with the bank's records

**~25 minutes · nothing to fill in · Run All takes about 2 minutes**

You built the Global Bank support desk as a crew in Labs 1 and 2. Now build it again in **Google ADK**
(Agent Development Kit). Lab 4 then compares two working versions of one desk.

This time the desk gets a second tool. It can read **what the bank's systems show** for the account:
the failed transfers, the fee that was charged, the change request waiting for approval. A reply built
on those records tells the customer what actually happened, not "we are looking into it".

ADK asks you to write some things that CrewAI hides: a **runner**, a **session**, and an **event
stream** that you read yourself. Watch for that too.

**Words used in this lab**

- **Runner:** the ADK object that runs an agent and sends back events.
- **Event:** one step of a run, for example a tool call, a tool result or an answer. The runner sends
  them one at a time.
- **Session:** the conversation so far, stored by a **session service**.
- **Session state:** named values saved in the session. One agent can write a value, and a later agent
  can read it.
- **Sub-agent:** an agent that runs inside another agent, for example as one step in a fixed order.

## 1 · The model, and the first thing that breaks

⚠️ **ADK is the opposite of CrewAI in three ways.**

- It needs the `openai/` prefix on the model name.
- It takes `api_base`, not `base_url`.
- Its support for OpenAI-style models needs an extra package, `google-adk[extensions]`. Your sandbox
  already has it.

If you copy the CrewAI setup cell from Lab 1 into this notebook, it does not work.

In [ ]:
import os
from google.adk.models.lite_llm import LiteLlm

MODEL = os.environ["OPENAI_MODEL"]
BASE  = os.environ["OPENAI_BASE_URL"]
KEY   = os.environ["OPENAI_API_KEY"]

def model():
    return LiteLlm(
        model    = f"openai/{MODEL}",   # <- prefix needed, unlike CrewAI
        api_base = BASE,                # <- api_base, not base_url
        api_key  = KEY,
    )

print("model id:", model().model)

## 2 · Two tools, and each is just a function

You do not need a decorator. ADK reads the function's **signature and docstring** to build the tool
description that the model sees. So the type hints and the docstring are the interface.

An ADK tool usually returns a `dict`, not a string. A dict gives the model named fields.

`account_activity` is the new tool. It returns what the bank's systems show for the account in a
ticket. Read what it returns for `GB-T-4474`. The customer's "missing" Rs 1,200 has an answer.

In [ ]:
from desk_kit import TICKETS, ACCOUNT_ACTIVITY, TEAM_RULE, print_queue, score

def ticket_lookup(ticket_id: str) -> dict:
    """Return the text of a Global Bank customer support ticket by its id, e.g. GB-T-4471."""
    return {"ticket_id": ticket_id,
            "text": TICKETS.get(ticket_id, f"No ticket found with id {ticket_id}")}

def account_activity(ticket_id: str) -> dict:
    """Return what the bank's systems show for the account in a ticket: payments, fees and requests."""
    return {"ticket_id": ticket_id,
            "records": ACCOUNT_ACTIVITY.get(ticket_id, "No records found")}

print(ticket_lookup("GB-T-4474"))
print(account_activity("GB-T-4474"))

## 3 · One agent, and the code to run it

Here ADK needs more setup than a crew. It helps to understand why.

- **`Runner`** runs the agent and sends back a stream of events.
- **`SessionService`** stores the conversation. ADK makes it an object that you can see and control.
- You **read the events yourself**. The `ask` function below picks out the final answer, and the
  names of the tools the agent called.

For a one-agent job this is extra work. It starts to help you in sections 5 and 6.

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

session_service = InMemorySessionService()

async def ask(runner, session_id, text):
    """Run one turn. Return the final answer and the names of the tools that were called."""
    await session_service.create_session(app_name="desk", user_id="u", session_id=session_id)
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    final, tools_called = None, []
    async for ev in runner.run_async(user_id="u", session_id=session_id, new_message=msg):
        tools_called += [call.name for call in ev.get_function_calls()]
        if ev.is_final_response() and ev.content and ev.content.parts:
            final = ev.content.parts[0].text
    return final, tools_called

triage = Agent(
    name="triage",
    model=model(),
    instruction=("Look up the ticket, then reply with 'Category: <x>' and 'Team: <y>' on separate lines. "
                 + TEAM_RULE),
    tools=[ticket_lookup],
)
runner = Runner(app_name="desk", agent=triage, session_service=session_service)

answer, tools_called = await ask(runner, "one-agent", "Classify ticket GB-T-4471.")
print(answer)
print("\ntools called:", tools_called)

> **You must use `await` here.** ADK's API is async first. `run_async` is the interface, and a
> notebook lets you `await` at the top level. ADK has no synchronous version that you could call by
> mistake. That is simpler than CrewAI, where `kickoff()` fails in a notebook.

## 4 · Named hand-offs, and records

This is how ADK passes work from one agent to the next:

1. An agent saves its answer in **session state** under a name. The name is its `output_key`.
2. The next agent's instruction reads that value back with a `{placeholder}` of the same name.

In a crew, the previous task's output arrives as plain context. **In ADK, the hand-off has a name**, and
you can see it in the code.

`output_schema` is ADK's version of CrewAI's `output_pydantic`. The classifier and the drafter both
answer with a record, using the same `Triage` and `DeskReply` shapes as the crew. An agent with an
`output_schema` cannot also call tools, so only the researcher has tools.

`build_desk` takes the researcher's tools as a setting. You build the desk twice in section 5.

In [ ]:
from pydantic import BaseModel, Field

class Triage(BaseModel):
    category: str = Field(description="The kind of problem, in a few words")
    team: str = Field(description="The team that owns the ticket")
    priority: str = Field(description="High, Medium or Low")

class DeskReply(BaseModel):
    customer_reply: str = Field(description="The reply the customer reads, at most four sentences")
    summary: str = Field(description="Handover: what happened, in one sentence")
    what_to_check: str = Field(description="Handover: what the owning team should check first")
    next_action: str = Field(description="Handover: the next thing the owning team should do")

RULES = ("Never promise a refund or credit, never give a deadline, "
         "and never ask for an OTP, PIN, CVV or password.")

def build_desk(research_tools):
    researcher = Agent(
        name="researcher", model=model(), tools=research_tools,
        instruction="Use your tools to look up everything about the ticket. List only the facts. "
                    "Do not interpret them.",
        output_key="facts",                                   # <- saves the answer in session state
    )
    classifier = Agent(
        name="classifier", model=model(), output_schema=Triage,
        instruction=("Facts:\n{facts}\n\n"                     # <- reads it back by name
                     "Classify the ticket. " + TEAM_RULE +
                     " The priority is High if the customer has lost money, otherwise Medium or Low."),
        output_key="triage",
    )
    drafter = Agent(
        name="drafter", model=model(), output_schema=DeskReply,
        instruction=("Facts:\n{facts}\nTriage:\n{triage}\n\n"
                     "Write the first reply to the customer, and a handover note for the owning team. "
                     "Tell the customer what the facts show. " + RULES),
        output_key="reply",
    )
    return SequentialAgent(name="desk", sub_agents=[researcher, classifier, drafter])

## 5 · Run the queue: ticket only, then with the records

`SequentialAgent` runs its sub-agents in order. The list is the wiring, like a crew's task order.

`run_queue` handles all five tickets, one session each. After each ticket it reads the session state:
`triage` and `reply` are the records the classifier and the drafter saved.

The first desk can only read the ticket, like the crew in Labs 1 and 2. The second desk can also read
the bank's records. Watch the `says what happened` column.

In [ ]:
import time
from google.adk.agents import SequentialAgent

async def run_queue(desk_agent, label):
    runner = Runner(app_name="desk", agent=desk_agent, session_service=session_service)
    rows, paths = [], {}
    start = time.time()
    for ticket_id in TICKETS:
        session_id = f"{label}-{ticket_id}"
        _, paths[ticket_id] = await ask(runner, session_id, f"Handle ticket {ticket_id}.")
        state = (await session_service.get_session(app_name="desk", user_id="u",
                                                   session_id=session_id)).state
        decision, reply = state["triage"], state["reply"]
        rows.append({"ticket": ticket_id, "team": decision["team"], "priority": decision["priority"],
                     "reply": reply["customer_reply"], "summary": reply["summary"],
                     "what_to_check": reply["what_to_check"], "next_action": reply["next_action"]})
    print(f"{label}: five tickets in {time.time() - start:.0f}s")
    return rows, paths

ticket_only, _ = await run_queue(build_desk([ticket_lookup]), "ticket-only")
with_records, paths = await run_queue(build_desk([ticket_lookup, account_activity]), "with-records")

print()
print_queue(with_records)
print()
print("ticket only :", score(ticket_only))
print("with records:", score(with_records))

> ⚠️ **You may see a deprecation warning here.** ADK 2.9.1 says `SequentialAgent` is deprecated, and
> that you should use `Workflow` instead. But in this version you **cannot import `Workflow` from
> `google.adk.agents`**. The warning itself says `Workflow` "cannot yet be used as an LlmAgent
> sub-agent". So `SequentialAgent` is still the right choice today. ADK changes quickly, so check the
> release notes whenever you upgrade it.
>
> You may also see `UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL` for each
> tool. It does not change the result.

## 6 · Read the difference as the customer would

The same ticket, answered by the two desks:

In [ ]:
for before, after in zip(ticket_only, with_records):
    print(after["ticket"], "-", TICKETS[after["ticket"]])
    print("  ticket only :", before["reply"])
    print("  with records:", after["reply"])
    print("  handover    :", after["next_action"])
    print()

**This is the value the records add.** On `GB-T-4474` the customer learns that the Rs 1,200 is the debit
card's annual fee plus GST. There is no need for a second ticket. On `GB-T-4472` the customer learns
that the new mobile number is waiting for signature verification. That is why the OTP goes to the old
phone. The owning team gets a handover note that says what to check, not just "please investigate".

A tool gives an agent facts it could not know. A better prompt cannot do that.

## 7 · Check the path, not only the answer

A reply can look right even when the agent took the wrong path. For example, it might write a
confident reply without reading the records. `ask` collected the tools that each ticket called. Check
that every ticket used both tools:

In [ ]:
for ticket_id, called in paths.items():
    ok = "account_activity" in called and "ticket_lookup" in called
    print(f"{ticket_id}  {'yes' if ok else 'NO ':<4} {called}")

This is the idea behind ADK's **tool trajectory** score. `adk eval` checks that an agent called the
expected tools, in the expected order, on a set of saved cases. You just did the same check by hand,
from the event stream.

### What to look at

- **The extra code.** The runner, the session and the event loop are more to set up than a crew. They
  also showed you the tools each ticket called. In a crew you would need to turn on logging to see that.
- **The hand-offs.** With `output_key` and `{facts}` you can point at exactly what the drafter received.
- **The reply.** If the drafter writes about itself ("Hello, I am the drafter...") and not to the
  customer, the fix is in the instruction, not in ADK.

### The other way to combine agents

`SequentialAgent` always runs the same order. Sometimes you want an agent to *decide* whether to ask
another agent. For that, wrap the other agent as a tool:

```python
from google.adk.tools import agent_tool
supervisor = Agent(name="supervisor", model=model(),
                   tools=[agent_tool.AgentTool(agent=classifier)])
```

This is ADK's version of delegation. Unlike a crew's manager, **you choose exactly which agents it can
reach.**

---

**Next:** Lab 4 runs the full crew and the full ADK desk on the same queue, and asks you to choose one.